# NB1 · Reaching the data

**Building Clinical Decision Support Systems with Generative AI Tools**  
Technology and Artificial Intelligence Literacy Training in Health Sciences · Akdeniz University · 18 September 2026

Prof. Dr. Utku Köse · Süleyman Demirel University, Department of Computer Engineering  
Director, Artificial Intelligence Application and Research Center (YAZEM) · utkukose@sdu.edu.tr

---


## What the workshop builds

Across six notebooks a working clinical decision support system is written. The system
is a computer program, and each notebook adds a layer to it.

| Notebook | Layer added |
|---|---|
| NB1 | Reaching the data and taking a first look |
| NB2 | Cleaning, preparing the information, splitting into training and test groups |
| NB3 | Building the model, teaching it and measuring how well it does |
| NB4 | The layer that justifies the model's decision |
| NB5 | Safety guardrails and the compliance report |
| NB6 | An interface used from a web browser |

**You do not need to know Python.** You will not write the code. Each step gives you a
prompt; you pass it to a generative AI tool (ChatGPT, Claude, Gemini), then paste the
code it returns into the blank cell and run it.

Understanding what the code does is expected. A short Python note follows each step and
explains the structures you will have seen in the code you received.


## How the notebook works

Each prompt ends with a section headed EXPECTED RESULT. It states what the code must
produce: which name will hold which piece of information. The check cell that follows
the paste cell tests exactly that.

When a check reports a shortcoming, return the code to the AI tool, state what the check
reported and have it regenerated. Not succeeding on the first attempt is normal.

The first line of each paste cell reads `#@cdss step_name`. **Do not delete that line.**
Paste your code below it. At the end of the notebook every step is collected into a
single block, which you carry to the next notebook.


## Setup

The cell below downloads the check helper. It is the only code handed to you in this
workshop; everything else you will have generated.


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib

import urllib.request

REPO = 'https://raw.githubusercontent.com/utkukose/cdss-genai-NB-lecture/main'
urllib.request.urlretrieve(f'{REPO}/workshop/cdss_kit.py', 'cdss_kit.py')

import cdss_kit as kit
kit.LANG = 'en'

print('Ready.')


---

## Choosing a dataset

Six datasets have been prepared for the workshop. All are openly accessible, require no
password and are reached from inside Colab. Choose one; every later step proceeds
according to your choice.

| Code | Data type | Dataset | Access | Condition to predict |
|---|---|---|---|---|
| `mimic-icu` | table | MIMIC-IV demo, intensive care records of 100 patients | Directly from the web | A stay exceeding three days |
| `wisconsin` | table | Breast Cancer Wisconsin, 569 samples | Bundled with scikit-learn | The mass being malignant |
| `pneumonia-mnist` | image | PneumoniaMNIST, 5,856 paediatric chest radiographs | `pip install medmnist` | Pneumonia present |
| `breast-mnist` | image | BreastMNIST, 780 breast ultrasound images | `pip install medmnist` | The mass being malignant |
| `mimic-ecg` | signal | MIMIC-IV-ECG demo, 659 ECGs from 92 patients | Directly from the web | A stay exceeding three days |
| `synthetic-notes` | text | Generated clinical notes | The code generates them | A condition you define |

### What to know before choosing

**Not every set carries a patient identifier.** In the MIMIC intensive care records a
patient may have several stays, and in the ECG set a patient may have several recordings;
the patient level split taught in NB2 has real bite in those two. The MedMNIST sets carry
no patient identifier and each image counts as a separate record. In Wisconsin each row is
a separate person. This is not a shortcoming but the structure of those sets, and it is
noted in the notebook.

**The ECG set and the intensive care set cover the same patients.** Choosing the ECG route
takes the target from the clinical demo, so you predict the same condition from the signal
instead. That makes the two routes directly comparable.

**The text route has no real data.** No clinical note collection is available without
credentialing; the MIMIC note module requires it. Notes are therefore generated on that
route, and the compliance report states this.

**MedMNIST licensing.** The sets are published under CC BY 4.0, with the exception of
DermaMNIST. Both sets offered here fall under CC BY 4.0.


---

## Step 1 · Defining the problem and starting the program

Every program begins with a few lines of preparation. Two things happen: the ready made
toolkits are called in, and the values that will not change are written once.

Those values describe your problem, and every later step reads them; the prompts for
loading, cleaning, converting and the interface all consult this definition. It is the
only place you need to change if you are working on a problem of your own.


### The prepared definition for your dataset

The table below holds a problem definition for each dataset. Find your row; you will use
these values when filling in prompt 1.

| Code | PROBLEM | DECISION_MOMENT | TARGET_DEFINITION | DATA_TYPE |
|---|---|---|---|---|
| `mimic-icu` | Predicting whether a patient admitted to intensive care will have a prolonged stay | Six hours after the patient enters intensive care | An intensive care stay exceeding three days | `table` |
| `wisconsin` | Predicting whether a breast mass is malignant | When the measurements from the fine needle aspirate are available | The mass being malignant | `table` |
| `pneumonia-mnist` | Predicting whether a paediatric chest radiograph shows pneumonia | Immediately after the radiograph is taken, before expert review | Pneumonia present on the radiograph | `image` |
| `breast-mnist` | Predicting whether a mass seen on breast ultrasound is malignant | At the moment the ultrasound image is acquired | The mass being malignant | `image` |
| `mimic-ecg` | Predicting from the ECG whether a patient admitted to intensive care will have a prolonged stay | At the moment the ECG is recorded | An intensive care stay exceeding three days | `signal` |
| `synthetic-notes` | A problem you define | A moment you define | A condition you define | `text` |

Two of the sets also need a numeric threshold. For `mimic-icu` and `mimic-ecg` the target
is a stay exceeding three days, and `mimic-icu` additionally uses a six hour decision
window. The other four need no such threshold, because the decision moment is the moment
the measurement is taken.


### Prompt 1

Fill in the bracketed places below from your own row in the table above.

```
I am starting to write a clinical decision support system. This first cell will be the
preparation section of the program.

Call in the standard toolkits needed for handling data and for machine learning.

Then define the values that describe my problem, writing beside each what it means:

  DATASET = '[dataset code]'
  DATA_TYPE = '[table, image, signal or text]'
  PROBLEM = '[the clinical problem I want to solve, one sentence]'
  DECISION_MOMENT = '[when the system produces output]'
  TARGET_DEFINITION = '[the condition to be predicted]'
  RANDOM_SEED = 42

If DATASET is 'mimic-icu' or 'mimic-ecg', also define:
  TARGET_THRESHOLD_DAYS = 3      a stay longer than this counts as the target
If DATASET is 'mimic-icu', additionally define:
  DECISION_WINDOW_HOURS = 6      hours until the decision moment
For the other datasets these two numbers are not defined; the decision moment is the
moment the measurement is taken.

Write the address of my data as well:
  DATA_ROOT = '[the data address, or synthetic if there is none]'

Print the versions of the toolkits you used. Print PROBLEM, DECISION_MOMENT,
TARGET_DEFINITION, DATASET and DATA_TYPE as well, so that what I am working on is
visible.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
After the cell has run, these names must be available:
  pd, np, PROBLEM, DECISION_MOMENT, TARGET_DEFINITION, DATASET, DATA_TYPE, DATA_ROOT,
  RANDOM_SEED
```


In [ ]:
#@cdss hazirlik
# Paste the generated code below this line.


### Check 1


In [ ]:
kit.check_defined('pd', 'np', 'PROBLEM', 'DECISION_MOMENT', 'TARGET_DEFINITION',
                  'DATASET', 'DATA_TYPE', 'DATA_ROOT', 'RANDOM_SEED')


### Python note · What is in the code you received

At the top of your code you will see lines such as `import pandas as pd`. A **library**
is a ready made toolkit written by others. `import` calls it into the program, and
`as pd` gives it a short name, so `pd` can be written instead of `pandas` from then on.
`pandas` is for working with tables and `numpy` for calculating with numbers.

The line `DECISION_WINDOW_HOURS = 6` defines a **variable**. Writing it in capitals tells
the reader that this value will not change during the program. Python does not enforce
this; it is a convention.

Keeping these values at the top matters. A `6` buried in the middle of the code becomes,
six months later, a value nobody remembers the meaning of. In a clinical system threshold
values are subject to audit and have to be visible in one place.

`RANDOM_SEED` serves this purpose: parts of machine learning involve randomness. Without
a fixed seed the program gives a slightly different result on each run, and which change
caused what can no longer be traced.


In [ ]:
# This cell is supplied. It prints the problem you defined.
print('Problem   :', PROBLEM)
print('Decision  :', DECISION_MOMENT)
print('Target    :', TARGET_DEFINITION)
print('Dataset   :', DATASET)
print('Data type :', DATA_TYPE)


---

## Step 2 · Bringing in the data

Use the prompt matching the dataset you chose. All six produce a result of the same shape:
a table containing `patient_id` and `target`. That shared shape is what lets every later
step proceed identically regardless of data type.

File addresses, column names and package names are written into the prompts. Do not expect
the AI tool to know them; it fills what it does not know by guessing, and the guess is
usually wrong. **You are the one who tells the tool what your data contains.** This is the
habit from the workshop that will serve you most.


### Prompt 2a · `mimic-icu` (table, shared scenario)

This prompt uses the `DECISION_WINDOW_HOURS` and `TARGET_THRESHOLD_DAYS` constants
from step 1.


```
Bring the data of patients admitted to intensive care into the program. The data is
openly accessible on the web and its address is defined above as DATA_ROOT.

There are three files, all compressed table files:
  {DATA_ROOT}/hosp/patients.csv.gz
      patient identifier (subject_id), sex (gender), age (anchor_age)
  {DATA_ROOT}/hosp/admissions.csv.gz
      patient identifier (subject_id), hospital admission number (hadm_id),
      type of admission (admission_type), insurance (insurance)
  {DATA_ROOT}/icu/icustays.csv.gz
      patient identifier (subject_id), hospital admission number (hadm_id),
      intensive care stay number (stay_id), which unit (first_careunit),
      time of entry (intime), time of exit (outtime),
      how many days the patient stayed in intensive care (los)

Do the following:
1. Start from the intensive care stays. Remove the patients who left before the decision
   moment, that is, those who stayed fewer than DECISION_WINDOW_HOURS hours.
2. Create the condition we will predict: 1 if the stay exceeds TARGET_THRESHOLD_DAYS,
   0 if not. Name it target.
3. Rename the patient identifier to patient_id.
4. Bring in the patient's sex, age, admission type and insurance from the other two
   files. The number of rows must not change during this; warn me if it does.
5. This item matters a great deal: remove from the result how many days the patient
   stayed (los) and when they left (outtime). Both are known only after the patient
   leaves and are not available at the decision moment.

Put all of this inside a piece of work named load_raw_data. Write an explanation of what
it does at its start. Then run it and keep the result under the name cohort. Show the
first five rows of the cohort.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a runnable piece of work named load_raw_data that returns a table.
There must be a table named cohort.
That table must contain patient_id, stay_id, target, gender, anchor_age,
admission_type, insurance and first_careunit.
That table must NOT contain los or outtime.
```




### Prompt 2b · `wisconsin` (table)

```
Bring the Breast Cancer Wisconsin dataset into the program. It ships with scikit-learn and
needs no download; load_breast_cancer opens it.

Do the following:
1. Load the set and turn it into a table.
2. Create the target under the name target: 1 if the mass is malignant, 0 if benign.
3. Each row belongs to a separate person; derive patient_id from the row order and note in
   a comment that this is not a real patient identifier.

Put this inside a piece of work named load_raw_data, write an explanation of what it does
at its start, run it and keep the result under the name cohort. Show the first five rows.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a table named cohort containing patient_id, target and the thirty
measurement columns.
```


### Prompt 2c · `pneumonia-mnist` or `breast-mnist` (image)

```
Bring an image set from the MedMNIST collection into the program.

Install with: pip install medmnist
Use PneumoniaMNIST where DATASET is 'pneumonia-mnist' and BreastMNIST where it is
'breast-mnist'. The images are 28x28 and single channel. The set arrives already divided
into training, validation and test; combine all three into one table, since we will do the
splitting ourselves in NB2.

Do the following:
1. Download and load the set.
2. Put each image into the table as one row.
3. Create the target under the name target; the set is already binary labelled.
4. This set carries no patient identifier. Derive patient_id from the row order and note
   in a comment that it is not a real identifier, so a patient level split cannot be
   verified here.
5. If the set is large, limit it to the first 2000 images and print that you did.

Put this inside a piece of work named load_raw_data, write an explanation of what it does
at its start, run it and keep the result under the name cohort. Show the first five rows.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a table named cohort containing:
  patient_id -> identifier derived from the row order
  image      -> the image itself
  target     -> 0 or 1
```


### Prompt 2d · `mimic-ecg` (signal)

```
Bring the ECG recordings from the MIMIC-IV-ECG demo into the program and take the target
from the MIMIC-IV clinical demo. The two sets cover the same 92 patients.

Install with: pip install wfdb

The ECG recordings are at https://physionet.org/files/mimic-iv-ecg-demo/0.1/
They are in WFDB format, ten seconds long, sampled at 500 Hz, with twelve leads. Each
patient's recordings sit in their own subdirectory named after the patient identifier. The
list of records is in record_list.csv.

For the target use the clinical demo:
  https://physionet.org/files/mimic-iv-demo/2.2/icu/icustays.csv.gz
  columns: subject_id, stay_id, los (days spent in intensive care)

Do the following:
1. Read record_list.csv and take the first 200 records; downloading all of them takes long.
2. Read each record with wfdb and keep the first lead only.
3. From the clinical demo find the longest intensive care stay for each patient. Set the
   target to 1 where it exceeds three days and 0 otherwise. Drop patients with no stay.
4. Keep the patient identifier as patient_id. A patient may have several recordings;
   print how many.

Put this inside a piece of work named load_raw_data, write an explanation of what it does
at its start, run it and keep the result under the name cohort. Show the first five rows.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a table named cohort containing:
  patient_id -> patient identifier, the same patient may have several rows
  signal     -> the ECG segment
  target     -> 0 or 1
```


### Prompt 2e · `synthetic-notes` (text)

```
Prepare data for a decision support system that works on clinical notes.

No clinical note collection is available without credentialing, so the notes are generated.

Use the values I defined above:
  PROBLEM           -> the clinical problem I want to solve
  TARGET_DEFINITION -> the condition to be predicted
  RANDOM_SEED       -> for reproducibility

Give the notes the features that make real notes hard: abbreviations, negation,
expressions of uncertainty, templated sentences repeated in every note, and sections copied
forward from earlier notes. The condition must not be recoverable from a single word; state
which surface cues you deliberately removed. Let the same patient have several notes.

Put this inside a piece of work named load_raw_data, write an explanation of what it does
at its start, run it and keep the result under the name cohort. Show the first five rows.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a table named cohort containing:
  patient_id -> patient identifier, the same patient may have several rows
  note       -> the clinical note
  target     -> 0 or 1
```


In [ ]:
#@cdss veri_yukleme
# Paste the generated code below this line.


### Check 2

Two checks run. The first looks at whether the piece of work runs, the second at whether
the resulting table matches the expected result. If you are not on the shared scenario,
change the column names in the second cell to match your own data.


In [ ]:
kit.check_function('load_raw_data', call_with=((), {}), expect_type=pd.DataFrame)


In [ ]:
# 'los' and 'outtime' exist only in the MIMIC intensive care set and are learned
# after the fact. The check is skipped for the other datasets.
forbidden = ['los', 'outtime'] if DATASET == 'mimic-icu' else []

kit.check_frame(cohort, name='cohort',
                required=['patient_id', 'target'], forbidden=forbidden, min_rows=30)


### Python note · Tables and pieces of work

In the code you will see an object named `cohort`. That object is a **table**; in the
Python world it is called a DataFrame. Think of an Excel sheet: rows hold records and
columns hold fields. What a row corresponds to depends on the set you chose: an intensive
care stay, a single image taken from a patient, an ECG recording or a clinical note. Two
columns are the same in every set: `patient_id` and `target`.

The line `def load_raw_data():` defines a **function**. A function is a piece of work with
a name. It is defined once and can be run as often as needed. Putting work into a function
brings three benefits: the work lives in one place, it can be repeated and it can be
tested.

The text in triple quotes just below the function is its **docstring**. Unlike a comment,
it can be read while the program runs, as the next cell shows.

`return` gives the result back. Without it the result stays inside the function and
nobody can use it.


In [ ]:
# Read the explanation written into the function.
help(load_raw_data)


In [ ]:
# The size of the table and its first rows.
print('Rows and columns:', cohort.shape)
print('Column names:', list(cohort.columns))
cohort.head()


### Leakage and the decision moment

The decision moment determines what information a system may use. Give the model
information that was not in the record at that moment and it performs almost perfectly,
because we have told it the answer. Once the system is deployed, that information is empty
at the decision moment and the system is of no use. This is called **leakage**.

It is the most common error in clinical code written with generative AI. It raises no
message, proceeds silently, and looks favourable at first because it improves the result.

Its form differs by dataset. In the MIMIC intensive care set, `los` and `outtime` are known
only after the patient leaves, which is why prompt 2a removes them explicitly. On the ECG
route the target comes from the clinical record, so those same columns are the target
itself and are never placed beside the signal. In the image sets a label hidden in a file
name or folder structure can do the same work. On the text route the sentence that states
the diagnosis outright is leakage.

Rather than catching leakage afterwards, we prevented it at the prompt stage. The
`forbidden` list in the check cell tested it on the MIMIC route as well.

The question to ask on the day you work with your own data is this: **was this column
populated at the decision moment?** Every column whose answer is no is removed.


---

## Step 3 · A first look at the data

Before moving to a model you have to know the table in front of you. In this step you
will have a piece of work written that summarises it.


### Prompt 3

```
Write a piece of work that summarises the cohort table. Name it summarise_data and let
it summarise whatever table it is given.

Have it print:
  - how many rows there are,
  - how many distinct patients there are,
  - the rate at which the target condition occurs, as a percentage.

Have it also give back a summary table with one row per column, containing:
  column       -> the name of the column
  type         -> the kind of information in it
  missing_rate -> how much of the column is empty
  distinct     -> how many different values it holds

Then run it on cohort, keep the result under the name summary and show it.

FORMAT
Write a single Python cell. Add a short comment beside each line. Keep it short and
readable; someone who does not know Python should be able to follow it. Show the
steps openly rather than using compact shortcuts. Below the code, summarise what it
does in three plain sentences.

EXPECTED RESULT
There must be a runnable piece of work named summarise_data that returns a table.
There must be a table named summary containing column, type, missing_rate and distinct.
```


In [ ]:
#@cdss veri_kesfi
# Paste the generated code below this line.


### Check 3


In [ ]:
kit.check_function('summarise_data', call_with=((cohort,), {}), expect_type=pd.DataFrame)


In [ ]:
kit.check_frame(summary, name='summary',
                required=['column', 'type', 'missing_rate', 'distinct'])


### Python note · Printing and giving back

The prompt asked for two different things: some figures printed, and a summary table
given back. The difference matters.

`print(...)` writes to the screen. What it writes stays on the screen and the program
cannot use it again. It cannot be tested either; the check cell cannot see text on a
screen.

`return ...` gives the result back. A table given back can be stored, tested and used in
later steps. That is why we had the summary table returned; you will add it to the
compliance report in NB5.

As a rule: print a result you only want a person to see, and return a result you want the
program to use.


### Reading the summary

Look at three things.

**Is the patient count lower than the row count?** If so, some patients have more than one
stay. You will have to take that into account when splitting the data in NB2.

**What is the rate of the target condition?** This is how often the condition we seek
occurs in the cohort. It will be the single most decisive figure when performance is
interpreted in NB3.

**Which columns are largely empty?** In NB2 you will decide how missing values are filled.
For now simply note them.


---

## End of notebook · Collecting the code

The cell below collects the code you carried from the earlier notebooks together with
what you added here, as one block. Copy the whole block; you will paste it into the
first cell of NB2.

The block is also saved as `cdss_nb1.py`. That file disappears when the Colab session closes,
so keep a copy in a text file on your own computer as well.


In [ ]:
code_so_far = kit.export(save_as='cdss_nb1.py')


## What this notebook did

The first layer of the system was written. The program now prepares itself, takes the
data from the web and summarises the table it holds.

Three habits were formed while the code was produced. What the data contains was stated
to the tool explicitly, and it was not left to guess. An expected result was written at
the end of every prompt, and the code that arrived was tested against it. Information not
available at the decision moment was removed while the data was still being loaded.

In NB2, cleaning, preparation of the information and the split into training and test
---

**Warning.** Nothing produced in this notebook is a validated clinical tool. The dataset
you used is an open collection prepared for teaching and does not represent the patient
population of your own institution. The material is for teaching.
